In [3]:
import matplotlib.pyplot as plt

from tick.plot import plot_hawkes_kernels
from tick.hawkes import SimuHawkesSumExpKernels, SimuHawkesMulti, \
    HawkesSumExpKern

In [ ]:

end_time = 1000
n_realizations = 10

decays = [.5, 2., 6.]
baseline = [0.12, 0.07]
adjacency = [[[0, .1, .4], [.2, 0., .2]],
             [[0, 0, 0], [.6, .3, 0]]]

hawkes_exp_kernels = SimuHawkesSumExpKernels(
    adjacency=adjacency, decays=decays, baseline=baseline,
    end_time=end_time, verbose=False, seed=1039)

In [ ]:

multi = SimuHawkesMulti(hawkes_exp_kernels, n_simulations=n_realizations)

multi.end_time = [(i + 1) / 10 * end_time for i in range(n_realizations)]
multi.simulate()

ZeroDivisionError: division by zero

In [ ]:
HawkesSumExpKern?

Init signature:
HawkesSumExpKern(
    decays,
    penalty='l2',
    C=1000.0,
    n_baselines=1,
    period_length=None,
    solver='agd',
    step=None,
    tol=1e-05,
    max_iter=100,
    verbose=False,
    print_every=10,
    record_every=10,
    elastic_net_ratio=0.95,
    random_state=None,
)
Docstring:     
Hawkes process learner for sum-exponential kernels with fixed and given
decays, with many choices of penalization and solvers.

Hawkes processes are point processes defined by the intensity:

.. math::
    \forall i \in [1 \dots D], \quad
    \lambda_i(t) = \mu_i(t) + \sum_{j=1}^D
    \sum_{t_k^j < t} \phi_{ij}(t - t_k^j)

where

* :math:`D` is the number of nodes
* :math:`\mu_i(t)` are the baseline intensities
* :math:`\phi_{ij}` are the kernels
* :math:`t_k^j` are the timestamps of all events of node :math:`j`

and with an sum-exponential parametrisation of the kernels

.. math::
    \phi_{ij}(t) = \sum_{u=1}^{U} \alpha^u_{ij} \beta^u
                   \exp (- \beta^u t) 1

In [15]:
learner = HawkesSumExpKern(decays, penalty='elasticnet',
                           elastic_net_ratio=0.8)
learner.fit(multi.timestamps)

AttributeError: 'HawkesSumExpKern' object has no settable attribute 'events'

In [7]:
fig = plot_hawkes_kernels(learner, hawkes=hawkes_exp_kernels, show=False)

for ax in fig.axes:
    ax.set_ylim([0., 1.])

NameError: name 'learner' is not defined

In [9]:
"""
=========================
Fit Hawkes random kernels
=========================

This Hawkes EM (`tick.inference.HawkesEM`) algorithm assume that kernels are
piecewise constant. Hence it can fit basically any kernel form. However it
doesn't scale very well.

It has been originally described in this paper:

Lewis, E., & Mohler, G. (2011).
A nonparametric EM algorithm for multiscale Hawkes processes.
`preprint, 1-16`_.

.. _preprint, 1-16: http://paleo.sscnet.ucla.edu/Lewis-Molher-EM_Preprint.pdf
"""

import numpy as np
import matplotlib.pyplot as plt

from tick.hawkes import (SimuHawkes, HawkesKernelTimeFunc, HawkesKernelExp,
                         HawkesEM)
from tick.base import TimeFunction
from tick.plot import plot_hawkes_kernels

run_time = 30000

t_values1 = np.array([0, 1, 1.5, 2., 3.5], dtype=float)
y_values1 = np.array([0, 0.2, 0, 0.1, 0.], dtype=float)
tf1 = TimeFunction([t_values1, y_values1],
                   inter_mode=TimeFunction.InterConstRight, dt=0.1)
kernel1 = HawkesKernelTimeFunc(tf1)

t_values2 = np.linspace(0, 4, 20)
y_values2 = np.maximum(0., np.sin(t_values2) / 4)
tf2 = TimeFunction([t_values2, y_values2])
kernel2 = HawkesKernelTimeFunc(tf2)

baseline = np.array([0.1, 0.3])

hawkes = SimuHawkes(baseline=baseline, end_time=run_time, verbose=False,
                    seed=2334)

hawkes.set_kernel(0, 0, kernel1)
hawkes.set_kernel(0, 1, HawkesKernelExp(.5, .7))
hawkes.set_kernel(1, 1, kernel2)

hawkes.simulate()

em = HawkesEM(4, kernel_size=16, n_threads=8, verbose=False, tol=1e-3)
em.fit(hawkes.timestamps)

fig = plot_hawkes_kernels(em, hawkes=hawkes, show=False)

for ax in fig.axes:
    ax.set_ylim([0, 1])
plt.show()

/Users/cherylkouadio/Documents/Repositories/Stage/.venv/lib/python3.14/site-packages/tick/hawkes/inference/hawkes_em.py:26: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
  * :math:`\mu_i` are the baseline intensities


AttributeError: 'History' object has no settable attribute '_minimum_col_width'